# Mental Health Prediction

This notebook provides a reproducible, leakage-safe baseline for the `treatment` target in the OSMI Mental Health in Tech survey. It intentionally does not ship data or precomputed results.

> This is an educational/research artifact, not a medical decision system. A missing or malformed dataset stops execution with an actionable error.


In [ ]:
import csv
import os
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
TEST_SIZE = 0.25
TARGET = "treatment"
DROP_COLUMNS = {"Country", "state", "Timestamp", "comments"}


## Load and validate the input

Set `MENTAL_HEALTH_DATA` to a CSV path. Without it, the notebook looks for `data/data.csv` relative to the repository root. The dataset is not committed to this repository.

In [ ]:
def resolve_data_path() -> Path:
    configured = os.environ.get("MENTAL_HEALTH_DATA")
    path = Path(configured).expanduser() if configured else Path("data") / "data.csv"
    if not path.is_file():
        raise FileNotFoundError(
            "Mental-health dataset not found. Put data.csv under data/ or set "
            "MENTAL_HEALTH_DATA to an existing CSV file; no training was run."
        )
    return path


def load_and_clean(path: Path) -> pd.DataFrame:
    try:
        frame = pd.read_csv(path, low_memory=False)
    except pd.errors.EmptyDataError as exc:
        raise ValueError(f"The dataset at {path} is empty.") from exc
    except pd.errors.ParserError as exc:
        raise ValueError(f"The dataset at {path} is not a valid CSV.") from exc
    try:
        with path.open(newline="", encoding="utf-8-sig") as handle:
            header = next(csv.reader(handle), [])
    except (OSError, csv.Error) as exc:
        raise ValueError(f"Unable to read the CSV header from {path}.") from exc
    if len(header) != len(set(header)):
        raise ValueError("Duplicate CSV column names are not supported.")
    if any(not isinstance(column, str) for column in frame.columns):
        raise ValueError("All CSV column names must be strings.")
    if frame.columns.duplicated().any():
        duplicates = frame.columns[frame.columns.duplicated()].tolist()
        raise ValueError(f"Duplicate CSV columns are not supported: {duplicates}")
    if frame.empty:
        raise ValueError(f"The dataset at {path} is empty.")
    if TARGET not in frame.columns:
        raise ValueError(
            f"Required target column {TARGET!r} is missing. "
            f"Available columns: {list(frame.columns)}"
        )

    # Remove free text, timestamp, and location fields that are not stable features.
    frame = frame.drop(columns=sorted(DROP_COLUMNS & set(frame.columns)))
    target = frame[TARGET].astype("string").str.strip().str.lower()
    unknown = sorted(set(target.dropna().unique()) - {"yes", "no"})
    if target.isna().any() or unknown:
        raise ValueError(
            "Target treatment must contain only Yes/No (apart from no missing values); "
            f"found unknown values={unknown} and missing={int(target.isna().sum())}."
        )
    frame[TARGET] = target.map({"yes": 1, "no": 0}).astype("int8")

    # Normalize whitespace while converting pandas' nullable string values to
    # ordinary object/NaN values supported by scikit-learn 1.2+.
    for column in frame.select_dtypes(include=["object", "string"]).columns:
        normalized = frame[column].astype("string").str.strip()
        frame[column] = normalized.astype(object).where(normalized.notna(), np.nan)
    if "Age" in frame.columns:
        age = pd.to_numeric(frame["Age"], errors="coerce")
        frame["Age"] = age.where(age.between(18, 100))
    all_missing = [column for column in frame.columns if column != TARGET and frame[column].isna().all()]
    if all_missing:
        frame = frame.drop(columns=all_missing)
    if len(frame.columns) == 1:
        raise ValueError("The dataset must contain at least one non-empty feature column besides treatment.")
    return frame


data_path = resolve_data_path()
dataset = load_and_clean(data_path)
print(f"Loaded {len(dataset):,} rows and {dataset.shape[1]:,} columns from {data_path}")
print(dataset[TARGET].value_counts(dropna=False).sort_index())


## Split before fitting preprocessing

The holdout split is stratified and fixed by `RANDOM_STATE`. Imputation, scaling, and one-hot encoding live inside each model pipeline, so statistics and category vocabularies are learned from training data only.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = dataset.drop(columns=[TARGET])
y = dataset[TARGET]
if X.shape[1] == 0:
    raise ValueError("The dataset has no usable feature columns after cleaning.")
if y.nunique() != 2 or y.value_counts().min() < 2:
    raise ValueError("Both treatment classes need at least two rows for a stratified holdout.")
if len(y) < 4:
    raise ValueError("At least four rows are required for a stratified train/test holdout.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [column for column in X_train.columns if column not in numeric_features]
print("train/test shapes:", X_train.shape, X_test.shape)
print("numeric features:", len(numeric_features), "categorical features:", len(categorical_features))


def make_preprocessor() -> ColumnTransformer:
    numeric = Pipeline(
        steps=[
            ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scale", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        steps=[
            ("impute", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric, numeric_features),
            ("categorical", categorical, categorical_features),
        ],
        remainder="drop",
    )


## Train and evaluate baselines

Accuracy is reported alongside balanced accuracy, precision, recall, F1, and ROC AUC because class imbalance can make accuracy misleading. These are holdout metrics, not claims of clinical performance.

In [ ]:
models = {
    "logistic_regression": LogisticRegression(
        solver="liblinear", max_iter=1000, random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"
    ),
}

fitted_models = {}
results = []
for name, estimator in models.items():
    pipeline = Pipeline(
        steps=[("preprocess", make_preprocessor()), ("model", estimator)]
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    results.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, predictions),
            "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
            "precision": precision_score(y_test, predictions, zero_division=0),
            "recall": recall_score(y_test, predictions, zero_division=0),
            "f1": f1_score(y_test, predictions, zero_division=0),
            "roc_auc": roc_auc_score(y_test, probabilities),
        }
    )
    fitted_models[name] = pipeline

# Preserve the pre-declared model order; do not rank or select models on the holdout.
metrics = pd.DataFrame(results).set_index("model")
metrics.round(4)


In [ ]:
# Inspect a pre-declared reference model; do not select a model using the holdout.
reference_name = "logistic_regression"
reference_model = fitted_models[reference_name]
holdout_predictions = pd.DataFrame(
    {"actual": y_test, "predicted": reference_model.predict(X_test)}
)
print(f"Reference model (not selected on holdout): {reference_name}")
holdout_predictions.head()


## Reproducibility and limitations

- No model or metric is generated when the dataset is absent; this repository contains no invented results.
- The split seed, model seeds, preprocessing order, and dependency ranges are explicit.
- The notebook reports both baselines and uses a pre-declared reference model; production model selection needs a validation scheme and a separately untouched test set.
- The model must not be used to diagnose, triage, or make decisions about a person.